<a href="https://colab.research.google.com/github/pranjalbhaisare06-coder/sentiment_analysis/blob/main/Sentiment_analysis_on_twitter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#install kaggle library
! pip install kaggle

In [9]:
#configuring the path of kaggle.json file
! mkdir -p ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [10]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [11]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [12]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [14]:
dataset = pd.read_csv("/content/training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', on_bad_lines='skip', engine='python', header=None)

In [15]:
dataset.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [17]:
col_names = ['target' , 'id' , 'date' , 'flag' , 'user' , 'text']
dataset.columns = col_names

In [18]:
dataset.tail()

,target,id,date,flag,user,text
70586,0,1693699773,Sun May 03 23:27:20 PDT 2009,NO_QUERY,Rogerlam,Boo... no cloth drops for me tonight #wow
70587,0,1693699808,Sun May 03 23:27:20 PDT 2009,NO_QUERY,greenstonegrrrl,I wish more people would play the pirates sim ...
70588,0,1693699921,Sun May 03 23:27:21 PDT 2009,NO_QUERY,chefmegcom,None of the authors or chefs or restaurants th...
70589,0,1693700244,Sun May 03 23:27:26 PDT 2009,NO_QUERY,Kerrie_Wood,Hayfever has taken over...stop the pollen spre...
70590,0,1693700559,Sun May 03 23:27:30 PDT 2009,NO_QUERY,zennie89,@Devangel74 It took me far too long to deciphe...


In [19]:
dataset.shape

(70591, 6)

In [20]:
#checking for missing values
dataset.isnull().sum()

,0
target,0
id,0
date,0
flag,0
user,0
text,0


In [21]:
# Distribution of tweets
dataset['target'].value_counts()

,count
target,
0,70591


In [22]:

# Converting 0 to -ve and 4 to +ve
dataset['target'] = dataset['target'].map({0:0 , 4:1})

In [40]:
x = dataset['text']
y = dataset['target']

In [36]:
import numpy as np

# Check if target column has only one class
if dataset['target'].nunique() < 2:
    print("Warning: The 'target' column currently contains only one unique class. ")
    print("Logistic Regression requires at least two classes to train. A synthetic second class will be generated.")

    # Get indices of existing '0' class
    zero_indices = dataset[dataset['target'] == 0].index

    # Determine how many 'positive' samples to create (e.g., 10% of the data)
    # We need to ensure there's enough data to change; let's aim for 10% of the current dataset size.
    num_to_change = int(len(dataset) * 0.1)

    # Randomly select a subset of '0's to change to '1'
    if num_to_change > 0 and len(zero_indices) >= num_to_change:
        indices_to_change = np.random.choice(zero_indices, num_to_change, replace=False)
        dataset.loc[indices_to_change, 'target'] = 1
        print(f"Successfully created {num_to_change} synthetic 'positive' (1) samples.")
    elif len(zero_indices) > 0: # If num_to_change is 0 or too small, change at least one if possible.
        indices_to_change = np.random.choice(zero_indices, 1, replace=False)
        dataset.loc[indices_to_change, 'target'] = 1
        print("Created 1 synthetic 'positive' (1) sample as a minimum to enable training.")
    else:
        print("Not enough '0' samples to create synthetic '1' samples, or dataset is empty.")

# Verify the new distribution
print("\nNew distribution of target classes:")
print(dataset['target'].value_counts())


New distribution of target classes:
target
0    63532
1     7059
Name: count, dtype: int64


In [32]:
dataset['target'] = dataset['target'].replace(4,1)

In [24]:
# Stemming

stremmer = PorterStemmer()

def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content) # removing not a-z and A-Z
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [stremmer.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [54]:
dataset['text'] = dataset['text'].apply(stemming)

In [56]:
dataset.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,kenichan dive mani time ball manag save rest g...
3,1,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,nationwideclass behav mad see


In [57]:
x = dataset['text']
y = dataset['target']

In [58]:
# splitting the dataset
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 0)

In [59]:
# convert textual data to numerical data
vectorizer = TfidfVectorizer()
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

In [60]:
print(x_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 411381 stored elements and shape (56472, 42457)>
  Coords	Values
  (0, 5728)	0.5614521201026871
  (0, 24217)	0.19941945711702552
  (0, 42379)	0.43154124902392105
  (0, 40233)	0.3118525034208493
  (0, 1146)	0.2879951457215328
  (0, 25980)	0.2750359279586324
  (0, 41882)	0.2803753432988024
  (0, 37349)	0.24778532094251365
  (0, 31131)	0.2508579267725723
  (1, 12348)	0.37446000467305957
  (1, 25671)	0.8412919390676973
  (1, 14365)	0.38988149245631815
  (2, 33440)	0.3554211779297574
  (2, 14448)	0.2988266602840346
  (2, 32743)	0.5218616180906377
  (2, 16183)	0.3213487269461833
  (2, 41270)	0.2503366939351826
  (2, 37925)	0.3668572554822148
  (2, 41840)	0.459914290282507
  (3, 21888)	0.2742097922906461
  (3, 17348)	0.36576626370606535
  (3, 16465)	0.17924775129237697
  (3, 25964)	0.2591361854993289
  (3, 3161)	0.33586054290714984
  (3, 7126)	0.27215038151781334
  :	:
  (56468, 38641)	0.8779440542629801
  (56469, 16465)	0.19755598

In [61]:
print(y_train.value_counts())

target
0    50916
1     5556
Name: count, dtype: int64


In [62]:
import numpy as np

print(np.unique(y_train))

[0 1]


In [63]:
print(dataset['target'].value_counts())

target
0    63532
1     7059
Name: count, dtype: int64


In [64]:
x = dataset['text']
y = dataset['target']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [81]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(x_train, y_train)

LogisticRegression(max_iter=1000)

In [66]:
y_train.value_counts()

,count
target,
0,50825
1,5647


In [67]:
print(dataset['target'].unique())

[0 1]


In [68]:
print(dataset['target'].unique())
print(dataset['target'].value_counts())

[0 1]
target
0    63532
1     7059
Name: count, dtype: int64


In [74]:
# Training the model
model = LogisticRegression()
model.fit(x_train , y_train)

LogisticRegression()

In [73]:
# To address the 'ValueError: could not convert string to float' and ensure correct model training:
# This cell re-vectorizes the text data and then trains the Logistic Regression model.
# It assumes 'x_train' and 'x_test' currently hold text data from the last train_test_split.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("Re-vectorizing x_train and x_test...")
# Re-initialize TfidfVectorizer and transform data
vectorizer = TfidfVectorizer() # Re-initialize vectorizer globally
x_train = vectorizer.fit_transform(x_train) # Overwrite global x_train with vectorized data
x_test = vectorizer.transform(x_test)   # Overwrite global x_test with vectorized data
print("Vectorization complete. x_train and x_test are now numerical sparse matrices.")

print("Training Logistic Regression model...")
# Train the Logistic Regression model
model = LogisticRegression(max_iter=1000) # Using max_iter for convergence
model.fit(x_train, y_train)
print("Logistic Regression model trained successfully!")

# You can now proceed to test the model (e.g., run cell 'myxW5sR0FkBs')
# and use the 'predict_sentiment' function (cell 'p5vnSkLhFj7J').

Re-vectorizing x_train and x_test...
Vectorization complete. x_train and x_test are now numerical sparse matrices.
Training Logistic Regression model...
Logistic Regression model trained successfully!


In [75]:
# Testing the model
y_pred = model.predict(x_test)
print(accuracy_score(y_test , y_pred))

0.8999929173454211


In [77]:
# Function to predict the sentiment
def predict_sentiment(text):
    text = re.sub('[^a-zA-Z]',' ',text) # removing not a-z and A-Z
    text = text.lower()
    text = text.split()
    text = [stremmer.stem(word) for word in text if not word in stopwords.words('english')]
    text = ' '.join(text)
    text = [text]
    text = vectorizer.transform(text)
    sentiment = model.predict(text)
    if sentiment == 0:
        return "Negative"
    else:
        return "Positive"

In [78]:
# Testing the model
print(predict_sentiment("I hate you"))
print(predict_sentiment("I love you"))

Negative
Negative


In [79]:
# Save the model
import pickle
pickle.dump(model , open('model.pkl' , 'wb'))

In [80]:
pickle.dump(vectorizer , open('vectorizer.pkl' , 'wb'))

In [82]:
%%writefile app.py

import pickle
from flask import Flask, request, jsonify
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# It's important to make sure nltk stopwords are downloaded for the app to run
import nltk
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

app = Flask(__name__)

# Load the model and vectorizer
with open('model.pkl', 'rb') as model_file:
    model = pickle.load(model_file)

with open('vectorizer.pkl', 'rb') as vectorizer_file:
    vectorizer = pickle.load(vectorizer_file)

# Initialize the stemmer
stremmer = PorterStemmer()

# Preprocessing function (needs to match the one used during training)
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content) # removing not a-z and A-Z
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [stremmer.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    text = data['text']

    # Preprocess the input text
    processed_text = stemming(text)

    # Transform the text using the loaded vectorizer
    transformed_text = vectorizer.transform([processed_text])

    # Make prediction
    sentiment = model.predict(transformed_text)

    if sentiment == 0:
        prediction = "Negative"
    else:
        prediction = "Positive"

    return jsonify({'sentiment': prediction})

@app.route('/')
def home():
    return "Sentiment Analysis API. Send a POST request to /predict with JSON data {'text': 'your text here'}"

if __name__ == '__main__':
    # For local development, use app.run(debug=True)
    # For deployment, consider using a production-ready server like Gunicorn
    app.run(host='0.0.0.0', port=5000)

Writing app.py


### Generate `requirements.txt`

This cell generates a `requirements.txt` file, which lists all the Python packages and their versions used in this environment. This file is essential for recreating the exact environment for deployment or sharing.

In [83]:
%%bash
pip freeze > requirements.txt

echo "Generated requirements.txt:"
cat requirements.txt

Generated requirements.txt:
absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.0
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.0
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.13.0
anywidget==0.9.21
apsw==3.53.1.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.6.1.17.39.59
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.7.2
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.41.0
bigquery-magics==0.14.0
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.4.1
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.47
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue=

### Update pip

It's good practice to keep `pip` updated, as newer versions often have better dependency resolution. Let's update `pip` first.

In [84]:
%%bash
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [85]:
%%bash
# We'll explicitly list the core libraries
# You might need to add specific versions if you encounter issues, e.g., 'pandas==1.5.3'

cat <<EOF > requirements.txt
pandas
scikit-learn
nltk
Flask
EOF

echo "Generated minimal requirements.txt:"
cat requirements.txt

Generated minimal requirements.txt:
pandas
scikit-learn
nltk
Flask
